# 02 — Data Cleaning

Runs the full cleaning pipeline (`src/clean_data.py`) against the raw data
and inspects the before/after state of each table plus the generated
**Data Quality Report**.


In [1]:

result = run_script("clean_data.py")

if result.returncode != 0:
    print("Data cleaning failed.")


Loading raw data...

Raw shapes:
  customers: (52520, 8)
  subscriptions: (52563, 11)
  transactions: (397803, 8)
  customer_activity: (976631, 7)
  support_tickets: (78233, 7)

=== Cleaning customers ===
  [Duplicate customers (duplicate customer_id)] affected=520 (0.99%) -> Deduplicated (kept first occurrence)
  [Inconsistent country capitalization] affected=2557 (4.917%) -> Standardized to title case
  [Inconsistent gender capitalization] affected=1040 (2.0%) -> Standardized to title case
  [Invalid age (<13 or >100)] affected=260 (0.5%) -> Converted to missing, then imputed
  [Missing age (incl. converted invalid ages)] affected=1338 (2.573%) -> Median imputation by region
  [Missing gender] affected=1056 (2.031%) -> Filled with 'Unknown' category
  [Missing country/region] affected=1070 (2.058%) -> Filled with 'Unknown' category
  [Missing referral_source] affected=1103 (2.121%) -> Filled with 'Not Captured'

=== Cleaning subscriptions ===
  [Inconsistent subscription_status label

## Before vs after: row counts

In [2]:

import pandas as pd

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

names = [
    "customers",
    "subscriptions",
    "transactions",
    "customer_activity",
    "support_tickets"
]

rows = []

for n in names:

    raw_n = len(
        pd.read_csv(RAW / f"{n}.csv")
    )

    clean_n = len(
        pd.read_csv(PROCESSED / f"{n}.csv")
    )

    rows.append(
        {
            "table": n,
            "raw_rows": raw_n,
            "processed_rows": clean_n,
            "removed": raw_n - clean_n
        }
    )

comparison = pd.DataFrame(rows)

print(
    comparison.to_string(index=False)
)


            table  raw_rows  processed_rows  removed
        customers     52520           52000      520
    subscriptions     52563           52563        0
     transactions    397803          393498     4305
customer_activity    976631          976631        0
  support_tickets     78233           78233        0


## Data Quality Report preview

In [3]:

report_path = ROOT / "reports" / "data_quality_report.md"

report_text = report_path.read_text(
    encoding="utf-8"
)

print(report_text[:3500])


# Data Quality Report

**Project:** Customer Retention Intelligence Platform (NovaStream)

This report documents every data quality issue discovered in the raw NovaStream datasets during the cleaning stage (`src/clean_data.py`), the number and percentage of records affected, the treatment applied, and the reasoning behind that treatment.

| Issue | Records Affected | % Affected | Treatment | Reason |
|---|---:|---:|---|---|
| Duplicate customers (duplicate customer_id) | 520 | 0.99% | Deduplicated (kept first occurrence) | Simulated double-entry during signup; customer_id is the natural key |
| Inconsistent country capitalization | 2,557 | 4.917% | Standardized to title case | Source systems submitted country names in mixed case |
| Inconsistent gender capitalization | 1,040 | 2.0% | Standardized to title case | Manual data entry inconsistency |
| Invalid age (<13 or >100) | 260 | 0.5% | Converted to missing, then imputed | Ages outside plausible human range are data entry errors |
| M

## Sanity check: no more invalid values after cleaning

In [4]:

customers = pd.read_csv(
    PROCESSED / "customers.csv"
)

subs = pd.read_csv(
    PROCESSED / "subscriptions.csv"
)

print(
    "Duplicate customer_id after cleaning:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Ages out of range after cleaning:",
    (
        (customers["age"] < 13)
        |
        (customers["age"] > 100)
    ).sum()
)

print(
    "Cancelled subs missing cancellation_date:",
    (
        (subs["subscription_status"] == "Cancelled")
        &
        (subs["cancellation_date"].isna())
    ).sum()
)

print(
    "Subscription status distinct values:",
    subs["subscription_status"].unique()
)


Duplicate customer_id after cleaning: 0
Ages out of range after cleaning: 0
Cancelled subs missing cancellation_date: 0
Subscription status distinct values: ['Active' 'Cancelled']
